# DIMAS-TI — Orang A, fokus 2026

Notebook ini menggunakan **data asli** dari `data/interim` dan **dataset gabungan baru** `data/processing/dataset_gabungan_2026.parquet` apabila ekspor GEE sudah selesai. Semua angka dan grafik dihitung dari berkas nyata; tidak ada nilai hujan/cuaca yang disimulasikan.

Cakupan data: enam provinsi, 1 Januari–16 Oktober 2026 (atau tanggal terakhir kedua sumber GEE, yang lebih awal), grid sekitar 0,045°. Label `y_7d` menggunakan FIRMS t+1 sampai t+7.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'data' / 'interim' / 'grid.csv').exists():
    ROOT = ROOT.parent
assert (ROOT / 'data' / 'interim' / 'grid.csv').exists(), 'Buka notebook dari folder karhutla'
RAW = ROOT / 'data' / 'raw'
INTERIM = ROOT / 'data' / 'interim'
OUTPUT = ROOT / 'data' / 'processing'
grid = pd.read_csv(INTERIM / 'grid.csv')
firms = pd.read_parquet(INTERIM / 'hotspots_daily.parquet')
firms['date'] = pd.to_datetime(firms['date'])
firms = firms.loc[firms.date.dt.year.eq(2026)].copy()
print('Grid total:', len(grid), '| FIRMS agregat 2026:', len(firms))
print('Provinsi:', ', '.join(sorted(grid.province.unique())))


## Audit data hotspot dan cakupan cuaca 2026

In [ ]:
display(firms.groupby(firms.date.dt.to_period('M'))['n_hotspot'].agg(['sum','count']).rename(columns={'sum':'detections','count':'active_grid_days'}))
summary = firms.groupby(firms.date.dt.to_period('M'))['n_hotspot'].sum()
ax = summary.plot.bar(figsize=(10,4), title='Total deteksi hotspot per bulan, 2026 (FIRMS)')
ax.set_ylabel('Jumlah deteksi'); ax.set_xlabel('Bulan'); plt.tight_layout(); plt.show()


In [ ]:
target = OUTPUT / 'dataset_gabungan_2026.parquet'
if target.exists():
    import pyarrow.parquet as pq
    meta = pq.read_metadata(target)
    print('Dataset gabungan tersedia:', target, 'baris:', meta.num_rows)
    import pyarrow.dataset as ds
    sample = ds.dataset(target).to_table(columns=['date','province','precip_mm','t2m_c','rh','sm1','wind_ms','y_7d']).slice(0,100000).to_pandas()
    print('Contoh missing cuaca (hingga 100 ribu baris):')
    display(sample.isna().mean().mul(100).round(2))
    display(sample.groupby('province')['y_7d'].agg(['mean','count']))
else:
    print('Belum ada dataset gabungan 2026. Jalankan proses GEE --run-2026, lalu --merge-2026. Tidak mengarang cuaca.')
